In [1]:
from google.colab import drive
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Kết nối Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Lệnh copy toàn bộ thư mục (sẽ mất khoảng 1-2 phút)
!cp -r "/content/drive/MyDrive/DATA_FACES" "/content/DATA_FACES_LOCAL"

print("Copy hoàn tất! Đã sẵn sàng ép xung!")

Copy hoàn tất! Đã sẵn sàng ép xung!


In [3]:


# 2. Đường dẫn trỏ thẳng ra ngoài MyDrive
# (Nếu bạn đặt tên khác thì thay chữ DATA_FACES thành tên của bạn nhé)
file_path = "/content/DATA_FACES_LOCAL"

# 3. Augmentation - Làm phong phú dữ liệu khuôn mặt
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# 4. Nạp ảnh
train_generator = train_datagen.flow_from_directory(
    file_path,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

print("\nDanh sách lớp:")
print(train_generator.class_indices)

Found 1925 images belonging to 31 classes.

Danh sách lớp:
{'BUI DANG KHOI': 0, 'DANG NGUYEN PHUONG NGHI': 1, 'HA PHUONG THAO': 2, 'HOANG BAO TRAN': 3, 'HOANG BUI TRA MY': 4, 'LE HUYNH DUC HUY': 5, 'LE MINH TRIET': 6, 'LE THAI BAO': 7, 'LE THI NHU QUYNH': 8, 'LE TRAN QUY ANH': 9, 'LE TRONG DAI': 10, 'MAI HO QUOC TUY': 11, 'NGUYEN BAO HAN': 12, 'NGUYEN DONG HAI': 13, 'NGUYEN HOANG BAO': 14, 'NGUYEN HUU TOAN': 15, 'NGUYEN KHAC LUU VU': 16, 'NGUYEN NGOC KHANH UYEN': 17, 'NGUYEN NGOC KIM TUYET': 18, 'NGUYEN THI THANH HA': 19, 'NGUYEN TRONG MINH': 20, 'NHAN MANH TUAN': 21, 'PHAM DUC THANH CONG': 22, 'PHAM LY BAO LAM': 23, 'PHAM MAI PHUONG': 24, 'THAI TUAN PHAT': 25, 'TRAN GIA HAN': 26, 'TRAN MINH HOANG': 27, 'TRAN NGOC THAO ANH': 28, 'TRAN THE DANG KHOA': 29, 'TRINH THUY HANG': 30}


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# ==========================================
# BƯỚC 3: XÂY DỰNG KIẾN TRÚC CNN (CONVOLUTIONAL NEURAL NETWORK)
# ==========================================
model = Sequential()

# Block 1: Trích xuất đặc trưng (Feature Extraction)
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))
model.add(MaxPooling2D(2, 2))

# Block 2
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

# Block 3
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))

# Chuyển đổi tensor 3D thành vector 1D (Flattening)
model.add(Flatten())

# Mạng Fully Connected (Dense Layers) để phân loại
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5)) # Drop 50% nơ-ron ngẫu nhiên để giảm Overfitting

# Lớp Output: 31 nơ-ron tương ứng với 31 classes (sử dụng hàm softmax để xuất xác suất)
model.add(Dense(31, activation='softmax'))

# Hiển thị tóm tắt kiến trúc mô hình và số lượng tham số
print("KIẾN TRÚC MÔ HÌNH:")
model.summary()


# ==========================================
# BƯỚC 4: BIÊN DỊCH VÀ HUẤN LUYỆN (COMPILE & TRAIN)
# ==========================================
# 1. Khai báo Optimizer, Loss function và Metrics
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy', # Hàm mất mát tiêu chuẩn cho Multi-class Classification
    metrics=['accuracy']
)

# 2. Thực thi quá trình huấn luyện
print("\nBẮT ĐẦU HUẤN LUYỆN MÔ HÌNH...")
history = model.fit(
    train_generator,
    epochs=30,
)

# 3. Xuất file mô hình (.h5) bao gồm cả kiến trúc và trọng số (weights)
model.save("/content/drive/MyDrive/face_model_31_classes.h5")
print("\n[SUCCESS] Đã lưu mô hình tại: /content/drive/MyDrive/face_model_31_classes.h5")

KIẾN TRÚC MÔ HÌNH:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 31)             │         3,999 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,308,639 (12.62 MB)

 Trainable params: 3,308,639 (12.62 MB)

 Non-trainable params: 0 (0.00 B)


BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH...
Epoch 1/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 582s 9s/step - accuracy: 0.0384 - loss: 3.4464
Epoch 2/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 102s 2s/step - accuracy: 0.0805 - loss: 3.3111
Epoch 3/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 103s 2s/step - accuracy: 0.1356 - loss: 3.0243
Epoch 4/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.2078 - loss: 2.7501
Epoch 5/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 106s 2s/step - accuracy: 0.2566 - loss: 2.5596
Epoch 6/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.3039 - loss: 2.3878
Epoch 7/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 105s 2s/step - accuracy: 0.3512 - loss: 2.2505
Epoch 8/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.3714 - loss: 2.1372
Epoch 9/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.4177 - loss: 1.9896
Epoch 10/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.4255 - loss: 1.8848
Epoch 11/30
61/61 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.4509 - loss: 1.8652
Epoch 12/30
61/61 ━━━━━━━━━━━━━━


[SUCCESS] Đã lưu mô hình tại: /content/drive/MyDrive/face_model_31_classes.h5
